# ABCA4 Advanced End-to-End Pipeline

This notebook implements the ABCA4 pipeline with leakage-aware cross-validation, ESM-2 delta embeddings, in-fold feature engineering/selection, calibration, MCC threshold optimization, AlphaMissense benchmarking, and VUS scoring.


In [ ]:
import os
import re
import json
import gzip
import pickle
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import requests
import shap
import xgboost as xgb
from scipy import stats
from transformers import AutoTokenizer, AutoModel
from imblearn.over_sampling import BorderlineSMOTE

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, matthews_corrcoef
from sklearn.isotonic import IsotonicRegression

try:
    from sklearn.frozen import FrozenEstimator
except Exception:
    class FrozenEstimator:
        def __init__(self, estimator):
            self.estimator = estimator
        def fit(self, X, y=None):
            return self
        def predict_proba(self, X):
            return self.estimator.predict_proba(X)
        def predict(self, X):
            return self.estimator.predict(X)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

ROOT = Path.cwd()
DATA_PATH = ROOT / 'ABCA4_mutations_annotated_with_features copy.csv'
WT_CACHE_PATH = ROOT / 'abca4_wt_sequence.txt'
ESM_CACHE_PATH = ROOT / 'esm_cache.pkl'
FEATURE_STABILITY_PATH = ROOT / 'feature_stability.csv'
MODEL_PATH = ROOT / 'abca4_binary_model.json'
CALIBRATOR_PATH = ROOT / 'isotonic_calibrator.pkl'
FEATURES_PATH = ROOT / 'features.json'
THRESHOLD_PATH = ROOT / 'threshold.json'
OOF_PATH = ROOT / 'oof_predictions.csv'
VUS_PRED_PATH = ROOT / 'vus_predictions.csv'
ALPHAMISSENSE_FILTERED_CACHE = ROOT / 'AlphaMissense_P78363_filtered.csv'

UNIPROT_FASTA_URL = 'https://rest.uniprot.org/uniprotkb/P78363.fasta'
ALPHAMISSENSE_URL = 'https://storage.googleapis.com/dm_alphamissense/AlphaMissense_aa_substitutions.tsv.gz'
ESM_MODEL_NAME = 'facebook/esm2_t33_650M_UR50D'


In [ ]:
# 1) Data loading + label normalization and VUS identification

df = pd.read_csv(DATA_PATH)
df['Significance_norm'] = df['Significance'].astype(str).str.strip().str.lower()

# Ambiguous/VUS strings used for VUS holdout
AMBIGUOUS_VUS_STRINGS = {
    'vus',
    'variant of uncertain significance',
    'uncertain significance',
    'uncertain',
    'ambiguous',
    'conflicting',
    'conflicting interpretations of pathogenicity',
    'unknown',
    'not provided'
}

BENIGN_STRINGS = {'benign', 'likely benign'}
PATHOGENIC_STRINGS = {'pathogenic', 'likely pathogenic'}

df['is_vus'] = df['Significance_norm'].isin(AMBIGUOUS_VUS_STRINGS)
df['binary_label'] = np.where(
    df['Significance_norm'].isin(BENIGN_STRINGS),
    0,
    np.where(df['Significance_norm'].isin(PATHOGENIC_STRINGS), 1, np.nan)
)

train_df = df[df['binary_label'].isin([0, 1])].copy()
train_df['binary_label'] = train_df['binary_label'].astype(int)
vus_df = df[df['is_vus']].copy()

print('Loaded shape:', df.shape)
print('Known labeled rows:', train_df.shape)
print('VUS rows:', vus_df.shape)
print('Significance_norm distribution:\n', df['Significance_norm'].value_counts(dropna=False))


In [ ]:
# Optional robustness diagnostics: class distributions + t-tests for numeric features
numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
exclude_num = {'binary_label', 'Position'}
numeric_cols = [c for c in numeric_cols if c not in exclude_num]

if numeric_cols:
    benign = train_df.loc[train_df['binary_label'] == 0, numeric_cols]
    patho = train_df.loc[train_df['binary_label'] == 1, numeric_cols]

    t_rows = []
    for col in numeric_cols:
        a = benign[col].dropna().values
        b = patho[col].dropna().values
        if len(a) > 2 and len(b) > 2:
            stat, p = stats.ttest_ind(a, b, equal_var=False, nan_policy='omit')
            t_rows.append((col, stat, p, np.nanmean(a), np.nanmean(b)))

    ttest_df = pd.DataFrame(t_rows, columns=['feature', 't_stat', 'p_value', 'mean_benign', 'mean_pathogenic'])
    ttest_df = ttest_df.sort_values('p_value', ascending=True)
    print('Top 15 t-test features:')
    display(ttest_df.head(15))
else:
    print('No numeric columns available for diagnostics.')


In [ ]:
# 2) Fetch/cache UniProt WT sequence for ABCA4 (P78363)

def load_wt_sequence(cache_path: Path, fasta_url: str) -> str:
    if cache_path.exists():
        seq = cache_path.read_text().strip().upper()
        if seq:
            return seq

    resp = requests.get(fasta_url, timeout=60)
    resp.raise_for_status()
    lines = [ln.strip() for ln in resp.text.splitlines() if ln.strip() and not ln.startswith('>')]
    seq = ''.join(lines).upper()
    if not seq:
        raise ValueError('Failed to parse sequence from UniProt FASTA response.')
    cache_path.write_text(seq + '\n')
    return seq

wt_sequence = load_wt_sequence(WT_CACHE_PATH, UNIPROT_FASTA_URL)
print('WT sequence length:', len(wt_sequence))
print('WT sequence cached at:', WT_CACHE_PATH)


In [ ]:
# 3) ESM-2 delta embeddings (mut - wt) for +/-20 AA window, cached to esm_cache.pkl

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch device:', device)

hf_token = os.getenv('HF_TOKEN')
if not hf_token:
    warnings.warn('HF_TOKEN is not set. If model download is gated/rate-limited, set HF_TOKEN in env.')

VARIANT_REGEX = re.compile(r'^([A-Za-z\*])([0-9]+)([A-Za-z\*])$')
WINDOW = 20
EMBED_DIM = 1280

def parse_variant(v: str):
    m = VARIANT_REGEX.match(str(v).strip())
    if not m:
        return None
    wt, pos, mut = m.group(1).upper(), int(m.group(2)), m.group(3).upper()
    return wt, pos, mut

def aa_clean(seq: str) -> str:
    allowed = set('ACDEFGHIKLMNPQRSTVWY')
    return ''.join(ch if ch in allowed else 'X' for ch in seq)

def get_window(seq: str, pos_1based: int, window: int = WINDOW):
    idx = pos_1based - 1
    start = max(0, idx - window)
    end = min(len(seq), idx + window + 1)
    local_idx = idx - start
    return seq[start:end], local_idx

def mutate_window(window_seq: str, local_idx: int, mut_aa: str) -> str:
    chars = list(window_seq)
    if 0 <= local_idx < len(chars):
        chars[local_idx] = mut_aa
    return ''.join(chars)

def load_esm_model(model_name: str, token: str | None, dev: torch.device):
    """Load ESM tokenizer/model on the selected device."""
    kwargs = {'token': token} if token else {}
    tokenizer = AutoTokenizer.from_pretrained(model_name, **kwargs)
    model = AutoModel.from_pretrained(model_name, **kwargs).to(dev).eval()
    return tokenizer, model

def embed_sequence(seq: str, tokenizer, model, dev):
    seq = aa_clean(seq)
    enc = tokenizer(seq, return_tensors='pt', add_special_tokens=True)
    enc = {k: v.to(dev) for k, v in enc.items()}
    with torch.no_grad():
        out = model(**enc).last_hidden_state
    token_emb = out[0, 1:-1, :]  # remove BOS/EOS
    if token_emb.shape[0] == 0:
        return np.zeros(EMBED_DIM, dtype=np.float32)
    return token_emb.mean(dim=0).detach().cpu().numpy().astype(np.float32)

def build_esm_cache(variants, wt_seq, cache_path: Path):
    if cache_path.exists():
        with open(cache_path, 'rb') as f:
            cache = pickle.load(f)
    else:
        cache = {}

    needed = [v for v in variants if v not in cache]
    print(f'ESM cache hits: {len(variants) - len(needed)} | misses: {len(needed)}')

    tokenizer = model = None
    if needed:
        tokenizer, model = load_esm_model(ESM_MODEL_NAME, hf_token, device)

    for i, v in enumerate(needed, 1):
        parsed = parse_variant(v)
        if parsed is None:
            cache[v] = np.zeros(EMBED_DIM, dtype=np.float32)
            continue

        wt_aa, pos, mut_aa = parsed
        if pos < 1 or pos > len(wt_seq):
            cache[v] = np.zeros(EMBED_DIM, dtype=np.float32)
            continue

        wt_window, center_idx = get_window(wt_seq, pos, WINDOW)
        if not wt_window:
            cache[v] = np.zeros(EMBED_DIM, dtype=np.float32)
            continue

        mut_window = mutate_window(wt_window, center_idx, mut_aa)
        wt_emb = embed_sequence(wt_window, tokenizer, model, device)
        mut_emb = embed_sequence(mut_window, tokenizer, model, device)
        cache[v] = (mut_emb - wt_emb).astype(np.float32)

        if i % 50 == 0 or i == len(needed):
            print(f'Computed {i}/{len(needed)} ESM deltas')

    with open(cache_path, 'wb') as f:
        pickle.dump(cache, f)

    return cache

all_variants = pd.concat([train_df['Variant'], vus_df['Variant']], axis=0).dropna().astype(str).unique().tolist()
esm_cache = build_esm_cache(all_variants, wt_sequence, ESM_CACHE_PATH)

esm_cols = [f'esm_delta_{i:04d}' for i in range(EMBED_DIM)]


def attach_esm_features(input_df: pd.DataFrame, cache: dict) -> pd.DataFrame:
    mat = np.vstack([cache.get(str(v), np.zeros(EMBED_DIM, dtype=np.float32)) for v in input_df['Variant'].astype(str)])
    esm_df = pd.DataFrame(mat, columns=esm_cols, index=input_df.index)
    return pd.concat([input_df.copy(), esm_df], axis=1)

train_df = attach_esm_features(train_df, esm_cache)
vus_df = attach_esm_features(vus_df, esm_cache)
print('train_df with ESM:', train_df.shape)
print('vus_df with ESM:', vus_df.shape)


In [ ]:
# 4) Cross-validation setup and helpers

PROTECTED_ALIASES = {
    'FunctionalDomain': ['FunctionalDomain'],
    'dHbond': ['dHbond', 'd_Hbond', 'Hbond'],
    'VDWClash': ['VDWClash']
}


def resolve_protected_features(df_: pd.DataFrame):
    resolved = []
    for target, aliases in PROTECTED_ALIASES.items():
        for c in aliases:
            if c in df_.columns:
                resolved.append(c)
                break
    return resolved


def preprocess_types(df_: pd.DataFrame, protected_features):
    out = df_.copy()
    if 'FunctionalDomain' in protected_features and 'FunctionalDomain' in out.columns:
        out['FunctionalDomain'] = out['FunctionalDomain'].astype('string').fillna('Unknown').astype('category')

    object_cols = [c for c in out.columns if out[c].dtype == 'object']
    for c in object_cols:
        if c == 'FunctionalDomain':
            continue
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out


def fit_imputation_values(df_train: pd.DataFrame):
    med = {}
    for c in df_train.columns:
        if pd.api.types.is_numeric_dtype(df_train[c]):
            med[c] = float(df_train[c].median()) if df_train[c].notna().any() else 0.0
    return med


def apply_imputation(df_in: pd.DataFrame, medians: dict, protected_features):
    out = df_in.copy()
    for c, v in medians.items():
        if c in out.columns:
            out[c] = out[c].fillna(v)
    if 'FunctionalDomain' in protected_features and 'FunctionalDomain' in out.columns:
        out['FunctionalDomain'] = out['FunctionalDomain'].astype('string').fillna('Unknown').astype('category')
    return out


def compute_vif(df_num: pd.DataFrame, cols: list[str]):
    """Compute per-feature variance inflation factor (VIF) for multicollinearity filtering."""
    vif = {}
    X_full = df_num[cols].to_numpy(dtype=float)
    for i, col in enumerate(cols):
        y = X_full[:, i]
        X = np.delete(X_full, i, axis=1)
        if X.shape[1] == 0:
            vif[col] = 1.0
            continue
        X = np.column_stack([np.ones(X.shape[0]), X])
        try:
            beta, *_ = np.linalg.lstsq(X, y, rcond=1e-15)
            y_hat = X @ beta
            ss_res = np.sum((y - y_hat) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            if ss_tot == 0:
                vif[col] = np.inf
            else:
                r2 = 1 - (ss_res / ss_tot)
                denom = max(1e-12, 1 - r2)
                vif[col] = float(1.0 / denom)
        except Exception:
            vif[col] = np.inf
    return vif


def vif_filter(train_df_fold: pd.DataFrame, candidate_cols: list[str], threshold: float = 10.0):
    cols = list(candidate_cols)
    dropped = []
    if not cols:
        return cols, dropped

    Xn = train_df_fold[cols].copy()
    for c in cols:
        Xn[c] = pd.to_numeric(Xn[c], errors='coerce')
        Xn[c] = Xn[c].replace([np.inf, -np.inf], np.nan)
        Xn[c] = Xn[c].fillna(Xn[c].median() if Xn[c].notna().any() else 0.0)

    while len(cols) > 1:
        vif_vals = compute_vif(Xn, cols)
        worst_col, worst_vif = max(vif_vals.items(), key=lambda kv: kv[1])
        if not np.isfinite(worst_vif) or worst_vif > threshold:
            dropped.append((worst_col, worst_vif))
            cols.remove(worst_col)
        else:
            break
    return cols, dropped


def sample_protected_for_smote(orig_train: pd.DataFrame, y_train: pd.Series, n_new: int, protected_features: list[str]):
    """Reconstruct protected features after SMOTE by minority-class sampling with replacement."""
    sampled = {}
    if n_new <= 0:
        for p in protected_features:
            sampled[p] = orig_train[p].reset_index(drop=True)
        return sampled

    class_counts = y_train.value_counts()
    minority_class = class_counts.idxmin()
    minority_idx = y_train[y_train == minority_class].index

    rng = np.random.default_rng(RANDOM_STATE)
    for p in protected_features:
        base = orig_train[p].reset_index(drop=True)
        if len(minority_idx) == 0:
            synth = pd.Series([base.mode(dropna=False).iloc[0] if not base.empty else np.nan] * n_new)
        else:
            pool = orig_train.loc[minority_idx, p].dropna()
            if pool.empty:
                fill_val = base.mode(dropna=False).iloc[0] if not base.empty else np.nan
                synth = pd.Series([fill_val] * n_new)
            else:
                picks = rng.choice(pool.values, size=n_new, replace=True)
                synth = pd.Series(picks)
        sampled[p] = pd.concat([base, synth], ignore_index=True)
    return sampled

metadata_drop = [
    'Variant', 'Significance', 'Significance_norm', 'Source', 'Annotation', 'binary_label', 'is_vus'
]
X_all = train_df.drop(columns=metadata_drop, errors='ignore').copy()
y_all = train_df['binary_label'].astype(int).copy()

protected_features = resolve_protected_features(X_all)
X_all = preprocess_types(X_all, protected_features)

esm_columns = [c for c in X_all.columns if c.startswith('esm_delta_')]
non_esm_columns = [c for c in X_all.columns if c not in esm_columns]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof_probs = np.zeros(len(train_df), dtype=float)
oof_fold = np.full(len(train_df), -1, dtype=int)
fold_thresholds = []
fold_selected_features = []
fold_vif_drops = []

print('X_all shape:', X_all.shape)
print('Protected features:', protected_features)
print('ESM columns:', len(esm_columns))


In [ ]:
# 5) In-fold pipeline: VIF -> BorderlineSMOTE (+protected handling) -> SHAP top-30 -> XGB + isotonic fold calibration -> MCC threshold

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all), start=1):
    print(f'\n===== Fold {fold} =====')
    X_tr_raw = X_all.iloc[tr_idx].copy()
    y_tr = y_all.iloc[tr_idx].copy()
    X_va_raw = X_all.iloc[va_idx].copy()
    y_va = y_all.iloc[va_idx].copy()

    # Step 1: VIF filtering on non-ESM, non-protected numeric features
    numeric_non_esm = [
        c for c in non_esm_columns
        if c in X_tr_raw.columns and c not in protected_features and pd.api.types.is_numeric_dtype(X_tr_raw[c])
    ]
    vif_keep, vif_drop = vif_filter(X_tr_raw, numeric_non_esm, threshold=10.0)
    drop_cols_vif = [c for c, _ in vif_drop]
    fold_vif_drops.append(vif_drop)

    fold_cols = [c for c in X_tr_raw.columns if (c in esm_columns) or (c in protected_features) or (c in vif_keep)]
    X_tr = X_tr_raw[fold_cols].copy()
    X_va = X_va_raw[fold_cols].copy()

    # Fit imputation from train fold only
    medians = fit_imputation_values(X_tr)
    X_tr = apply_imputation(X_tr, medians, protected_features)
    X_va = apply_imputation(X_va, medians, protected_features)

    # Step 2: BorderlineSMOTE on non-protected columns; protected sampled with replacement from minority class
    smote_input_cols = [c for c in X_tr.columns if c not in protected_features]
    X_smote_input = X_tr[smote_input_cols].copy()

    smote = BorderlineSMOTE(random_state=RANDOM_STATE, kind='borderline-1')
    X_res_np, y_res = smote.fit_resample(X_smote_input, y_tr)
    X_res = pd.DataFrame(X_res_np, columns=smote_input_cols)

    n_new = len(y_res) - len(y_tr)
    protected_resampled = sample_protected_for_smote(X_tr, y_tr, n_new, protected_features)
    for p in protected_features:
        X_res[p] = protected_resampled[p].values

    if 'FunctionalDomain' in protected_features and 'FunctionalDomain' in X_res.columns:
        X_res['FunctionalDomain'] = X_res['FunctionalDomain'].astype('string').fillna('Unknown').astype('category')
        X_va['FunctionalDomain'] = X_va['FunctionalDomain'].astype('string').fillna('Unknown').astype('category')

    # Step 3: Preliminary XGBoost + SHAP top 30 + protected
    prelim = xgb.XGBClassifier(
        n_estimators=250,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='binary:logistic',
        tree_method='hist',
        enable_categorical=True,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    prelim.fit(X_res, y_res)

    explainer = shap.TreeExplainer(prelim)
    shap_vals = explainer.shap_values(X_res)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[-1]

    mean_abs = np.abs(shap_vals).mean(axis=0)
    shap_rank = pd.Series(mean_abs, index=X_res.columns).sort_values(ascending=False)
    top30 = shap_rank.head(30).index.tolist()

    selected_features = list(dict.fromkeys(top30 + protected_features))
    fold_selected_features.append(selected_features)

    X_res_sel = X_res[selected_features].copy()
    X_va_sel = X_va[selected_features].copy()

    # Step 4: XGBoost + fold calibration with FrozenEstimator + isotonic
    model_fold = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='binary:logistic',
        tree_method='hist',
        enable_categorical=True,
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model_fold.fit(X_res_sel, y_res)

    calibrator_fold = CalibratedClassifierCV(
        FrozenEstimator(model_fold),
        method='isotonic',
        cv='prefit'
    )
    calibrator_fold.fit(X_va_sel, y_va)

    fold_probs = calibrator_fold.predict_proba(X_va_sel)[:, 1]
    oof_probs[va_idx] = fold_probs
    oof_fold[va_idx] = fold

    # Step 5: MCC threshold optimization on validation fold (0.05..0.95)
    thresholds = np.arange(0.05, 0.951, 0.01)
    mcc_scores = [matthews_corrcoef(y_va, (fold_probs >= t).astype(int)) for t in thresholds]
    best_i = int(np.argmax(mcc_scores))
    best_thr = float(thresholds[best_i])
    best_mcc = float(mcc_scores[best_i])
    fold_thresholds.append(best_thr)

    print(f'Fold {fold} | dropped by VIF: {len(drop_cols_vif)} | selected features: {len(selected_features)} | best_thr={best_thr:.2f} | best_mcc={best_mcc:.4f}')


In [ ]:
# 6) Post-fold feature stability (>=3/5 folds) + protected feature append
feature_counts = Counter(f for fold_feats in fold_selected_features for f in fold_feats)
stable_features = [f for f, c in feature_counts.items() if c >= 3]
for p in protected_features:
    if p not in stable_features and p in X_all.columns:
        stable_features.append(p)

feature_stability = (
    pd.DataFrame({'feature': list(feature_counts.keys()), 'count': list(feature_counts.values())})
      .sort_values(['count', 'feature'], ascending=[False, True])
)
feature_stability['selected_ge_3'] = feature_stability['count'] >= 3
feature_stability.to_csv(FEATURE_STABILITY_PATH, index=False)

print('Stable feature count:', len(stable_features))
print('Saved:', FEATURE_STABILITY_PATH)
display(feature_stability.head(40))


In [ ]:
# 7) Final model on all train data + isotonic calibration from OOF + artifact export

X_final = X_all[stable_features].copy()
final_medians = fit_imputation_values(X_final)
X_final = apply_imputation(X_final, final_medians, protected_features)

final_model = xgb.XGBClassifier(
    n_estimators=700,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='binary:logistic',
    tree_method='hist',
    enable_categorical=True,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
final_model.fit(X_final, y_all)

isotonic_calibrator = IsotonicRegression(out_of_bounds='clip')
isotonic_calibrator.fit(oof_probs, y_all)
oof_probs_cal = isotonic_calibrator.predict(oof_probs)

mean_threshold = float(np.mean(fold_thresholds)) if fold_thresholds else 0.5

final_model.save_model(str(MODEL_PATH))
with open(CALIBRATOR_PATH, 'wb') as f:
    pickle.dump(isotonic_calibrator, f)
with open(FEATURES_PATH, 'w') as f:
    json.dump(stable_features, f, indent=2)
with open(THRESHOLD_PATH, 'w') as f:
    json.dump({'mean_threshold': mean_threshold, 'fold_thresholds': fold_thresholds}, f, indent=2)

# Save OOF predictions
oof_df = train_df[['Variant', 'binary_label']].copy()
oof_df['fold'] = oof_fold
oof_df['oof_prob'] = oof_probs
oof_df['oof_prob_calibrated'] = oof_probs_cal
oof_df['pred_at_mean_thr'] = (oof_probs_cal >= mean_threshold).astype(int)
oof_df.to_csv(OOF_PATH, index=False)

# Final validation metrics on OOF-calibrated probabilities
auroc = roc_auc_score(y_all, oof_probs_cal)
mcc = matthews_corrcoef(y_all, (oof_probs_cal >= mean_threshold).astype(int))
brier = brier_score_loss(y_all, oof_probs_cal)

print('=== Final OOF Validation Metrics ===')
print(f'AUROC: {auroc:.4f}')
print(f'MCC (mean threshold={mean_threshold:.3f}): {mcc:.4f}')
print(f'Brier score: {brier:.4f}')
print(f'Mean threshold: {mean_threshold:.4f}')

print('Saved artifacts:')
print('-', MODEL_PATH)
print('-', CALIBRATOR_PATH)
print('-', FEATURES_PATH)
print('-', THRESHOLD_PATH)
print('-', OOF_PATH)


In [ ]:
# 8) AlphaMissense benchmark: stream/download, filter uniprot_id == P78363, compare AUROC + MCC

def load_alphamissense_abca4(cache_path: Path):
    if cache_path.exists():
        return pd.read_csv(cache_path)

    chunks = []
    usecols = ['uniprot_id', 'protein_variant', 'am_pathogenicity']
    for chunk in pd.read_csv(ALPHAMISSENSE_URL, sep='	', compression='gzip', chunksize=500_000, usecols=usecols):
        sub = chunk[chunk['uniprot_id'] == 'P78363'].copy()
        if not sub.empty:
            chunks.append(sub)

    if not chunks:
        raise ValueError('No ABCA4 rows found in AlphaMissense stream.')

    out = pd.concat(chunks, axis=0, ignore_index=True)
    out.to_csv(cache_path, index=False)
    return out

try:
    alpha_df = load_alphamissense_abca4(ALPHAMISSENSE_FILTERED_CACHE)
except Exception as e:
    print('Streaming AlphaMissense failed:', e)
    fallback = ROOT / 'ABCA4_alphamissense_scores.csv'
    if fallback.exists():
        alpha_df = pd.read_csv(fallback)
        if 'protein_variant' not in alpha_df.columns and 'Variant' in alpha_df.columns:
            alpha_df['protein_variant'] = alpha_df['Variant']
        if 'am_pathogenicity' not in alpha_df.columns and 'AlphaMissense_score' in alpha_df.columns:
            alpha_df['am_pathogenicity'] = alpha_df['AlphaMissense_score']
    else:
        raise

alpha_df = alpha_df.rename(columns={'protein_variant': 'Variant', 'am_pathogenicity': 'AlphaMissense_score'})
alpha_df = alpha_df[['Variant', 'AlphaMissense_score']].dropna().copy()

bench_df = oof_df.merge(alpha_df, on='Variant', how='inner')
bench_df['AlphaMissense_score'] = pd.to_numeric(bench_df['AlphaMissense_score'], errors='coerce')
bench_df = bench_df.dropna(subset=['AlphaMissense_score'])

if bench_df.empty:
    print('No overlap rows found for AlphaMissense benchmark.')
else:
    model_auroc = roc_auc_score(bench_df['binary_label'], bench_df['oof_prob_calibrated'])
    alpha_auroc = roc_auc_score(bench_df['binary_label'], bench_df['AlphaMissense_score'])

    model_mcc = matthews_corrcoef(bench_df['binary_label'], (bench_df['oof_prob_calibrated'] >= mean_threshold).astype(int))
    alpha_mcc = matthews_corrcoef(bench_df['binary_label'], (bench_df['AlphaMissense_score'] >= 0.5).astype(int))

    print('=== AlphaMissense Benchmark ===')
    print(f'Overlap rows: {len(bench_df)}')
    print(f'Model AUROC: {model_auroc:.4f}')
    print(f'AlphaMissense AUROC: {alpha_auroc:.4f}')
    print(f'Model MCC: {model_mcc:.4f}')
    print(f'AlphaMissense MCC (@0.5): {alpha_mcc:.4f}')


In [ ]:
# 9) VUS scoring with final model + exports

X_vus = vus_df.drop(columns=metadata_drop, errors='ignore').copy()
for f in stable_features:
    if f not in X_vus.columns:
        X_vus[f] = np.nan
X_vus = X_vus[stable_features]
X_vus = preprocess_types(X_vus, protected_features)
X_vus = apply_imputation(X_vus, final_medians, protected_features)

vus_probs_raw = final_model.predict_proba(X_vus)[:, 1]
vus_probs_cal = isotonic_calibrator.predict(vus_probs_raw)

vus_out = vus_df[['Variant', 'Significance']].copy()
vus_out['P_pathogenic_raw'] = vus_probs_raw
vus_out['P_pathogenic_calibrated'] = vus_probs_cal
vus_out['Classification_0.90_0.10'] = np.where(
    vus_probs_cal >= 0.90,
    'Likely Pathogenic',
    np.where(vus_probs_cal <= 0.10, 'Likely Benign', 'Remain VUS')
)

vus_out.to_csv(VUS_PRED_PATH, index=False)
print('Saved:', VUS_PRED_PATH)
print(vus_out['Classification_0.90_0.10'].value_counts())

print('Saved OOF predictions:', OOF_PATH)
